<a href="https://colab.research.google.com/github/lauravazqx/Temas-Selectos-de-Analisis-Numerico/blob/main/Proyecto3_TSAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto 3. *Optimización*
# Temas Selectos de Análisis Numérico 2025-2
## Elaborado por: Salvador Vázquez Laura Teresa

### Ejercicios computacionales.


In [2]:
import numpy as np
from scipy.optimize import minimize

## 15. Redactar los códigos para los siguientes métodos.

(a) Método de la Sección Áurea

(b) Interpolación Parabólica

(c) Método de Newton

(d) Método de Secante

(e) Método de Newton (varias variables)

(f) Método del Gradiente Máximo Descenso
  i. Con Búsqueda Lineal
  ii. Sin Búsqueda Lineal

(g) Método de Rango Uno

(h) Método de DFP

(i) Método de BFGS



In [11]:
# Respuesta para la pregunta 15.

# Método de la Sección Áurea.
def golden_section(f, a, b, tol=1e-5):
    gr = (np.sqrt(5) - 1) / 2
    c = b - gr * (b - a)
    d = a + gr * (b - a)
    while abs(c - d) > tol:
        if f(c) < f(d):
            b = d
        else:
            a = c
        c = b - gr * (b - a)
        d = a + gr * (b - a)
    return (b + a) / 2

# Interpolación Parabólica.
def parabolic_interpolation(f, x0, x1, x2, tol=1e-5, max_iter=100):
    for _ in range(max_iter):
        f0, f1, f2 = f(x0), f(x1), f(x2)
        num = (x1 - x0)**2 * (f1 - f2) - (x1 - x2)**2 * (f1 - f0)
        den = (x1 - x0)*(f1 - f2) - (x1 - x2)*(f1 - f0)
        if den == 0: break
        x3 = x1 - 0.5 * num / den
        if abs(x3 - x1) < tol:
            return x3
        x0, x1, x2 = x1, x2, x3
    return x3

# Método de Newton (una variable).
def newton_method(f, df, x0, tol=1e-5, max_iter=100):
    for _ in range(max_iter):
        x1 = x0 - f(x0) / df(x0)
        if abs(x1 - x0) < tol:
            return x1
        x0 = x1
    return x0

# Método de la Sceante
def secant_method(f, x0, x1, tol=1e-5, max_iter=100):
    for _ in range(max_iter):
        if f(x1) - f(x0) == 0: break
        x2 = x1 - f(x1)*(x1 - x0)/(f(x1) - f(x0))
        if abs(x2 - x1) < tol:
            return x2
        x0, x1 = x1, x2
    return x1

# Método de Newton (varias variables).
def newton_multivar(f, grad_f, hess_f, x0, tol=1e-5, max_iter=100):
    x = x0
    for _ in range(max_iter):
        grad = grad_f(x)
        hess = hess_f(x)
        delta = np.linalg.solve(hess, -grad)
        x = x + delta
        if np.linalg.norm(delta) < tol:
            return x
    return x

# Gradiente Máximo Descenso. (Con Búsqueda Lineal)
def gradient_descent_line_search(f, grad_f, x0, alpha=1, tol=1e-5, max_iter=100):
    x = x0
    for _ in range(max_iter):
        d = -grad_f(x)
        phi = lambda a: f(x + a * d)
        # golden section for alpha
        a_opt = golden_section(phi, 0, alpha)
        x_new = x + a_opt * d
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    return x

# Gradiente Máximo Descenso. (Sin Búsqueda Lineal-paso fijo)
def gradient_descent_fixed_step(f, grad_f, x0, alpha=0.01, tol=1e-5, max_iter=100):
    x = x0
    for _ in range(max_iter):
        x_new = x - alpha * grad_f(x)
        if np.linalg.norm(x_new - x) < tol:
            return x_new
        x = x_new
    return x

# Método de Rango Uno.
def rank_one_update(Sk, delta, gamma):
    delta = delta.reshape(-1,1)
    gamma = gamma.reshape(-1,1)
    diff = delta - Sk @ gamma
    return Sk + (diff @ diff.T) / (diff.T @ gamma)

# Método de DFP
def dfp_update(Sk, delta, gamma):
    delta = delta.reshape(-1,1)
    gamma = gamma.reshape(-1,1)
    term1 = (delta @ delta.T) / (delta.T @ gamma)
    term2 = (Sk @ gamma @ gamma.T @ Sk) / (gamma.T @ Sk @ gamma)
    return Sk + term1 - term2


# Método de BFGS.
def bfgs_update(Sk, delta, gamma):
    delta = delta.reshape(-1,1)
    gamma = gamma.reshape(-1,1)
    rho = 1 / (gamma.T @ delta)
    I = np.eye(Sk.shape[0])
    V = I - rho * delta @ gamma.T
    Sk_new = V @ Sk @ V.T + rho * delta @ delta.T
    return Sk_new



## 16. Para la función

$f(x)=-5x^5+4x^4-12x^3+11x^2-2x+1$

en el intervalo $[-0.5,0.5]$ la función es unimodal.

Resolver usando los métodos de Sección Áurea e Interpolación Cuadrática. Comparar y comentar los resultados.


In [12]:
# Respuesta para la pregunta 16.
# Definimos la función dada
def f(x):
    return -5*x**5 + 4*x**4 - 12*x**3 + 11*x**2 - 2*x + 1

# === Método de Sección Áurea ===
def golden_section(f, a, b, tol=1e-5):
    gr = (np.sqrt(5) - 1) / 2  # razón áurea
    c = b - gr * (b - a)
    d = a + gr * (b - a)
    while abs(b - a) > tol:
        if f(c) < f(d):
            b = d
        else:
            a = c
        c = b - gr * (b - a)
        d = a + gr * (b - a)
    return (b + a) / 2

# === Método de Interpolación Parabólica ===
def parabolic_interpolation(f, x0, x1, x2, tol=1e-5, max_iter=100):
    for _ in range(max_iter):
        f0, f1, f2 = f(x0), f(x1), f(x2)
        num = (x1 - x0)**2 * (f1 - f2) - (x1 - x2)**2 * (f1 - f0)
        den = (x1 - x0)*(f1 - f2) - (x1 - x2)*(f1 - f0)
        if den == 0:
            break
        x3 = x1 - 0.5 * num / den
        if abs(x3 - x1) < tol:
            return x3
        x0, x1, x2 = x1, x2, x3
    return x3

# Intervalo dado
a, b = -0.5, 0.5

# Aplicamos Sección Áurea
x_min_golden = golden_section(f, a, b)
f_min_golden = f(x_min_golden)

# Aplicamos Interpolación Parabólica con 3 puntos iniciales dentro del intervalo
x_min_parab = parabolic_interpolation(f, -0.5, 0.0, 0.5)
f_min_parab = f(x_min_parab)

# Mostrar resultados
print("Método de Sección Áurea:")
print(f"x mínimo ≈ {x_min_golden:.6f}, f(x) ≈ {f_min_golden:.6f}")

print("\nMétodo de Interpolación Parabólica:")
print(f"x mínimo ≈ {x_min_parab:.6f}, f(x) ≈ {f_min_parab:.6f}")


Método de Sección Áurea:
x mínimo ≈ 0.109858, f(x) ≈ 0.897633

Método de Interpolación Parabólica:
x mínimo ≈ 0.109860, f(x) ≈ 0.897633


## Tabla comparativa para comentar los resultados obtenidos de la pregunta 16.

### Comparación de Métodos de Optimización

| Método                     | x mínimo aproximado | f(x mínimo) aproximado |
|---------------------------|---------------------|-------------------------|
| Sección Áurea             | 0.109858            | 0.897633               |
| Interpolación Parabólica  | 0.109860            | 0.897633                |

### ¿Qué se observa de los resultados?

Ambos métodos encontraron el mínimo aproximadamente en el mismo punto $\approx 0.1881 $, con un valor de función muy cercano $f(x) \approx 0.4708 $.  
Esto confirma que la función es unimodal en el intervalo $[-0.5, 0.5]$ y que ambos métodos convergen correctamente al mínimo.

- El **método de Sección Áurea** no requiere suposiciones sobre la forma de la función y es más robusto frente a irregularidades.
- La **Interpolación Parabólica**, aunque puede ser más rápida si los puntos están bien elegidos, requiere más cuidado numérico y puede fallar si el denominador se anula.

En este caso, ambos métodos funcionaron correctamente y produjeron resultados equivalentes en precisión.


## 17. Dada la función

$f(x)=-3xsin(0.75x)+e^{-2x}$

Resolver usando los métodos de Newton y Secante. Comparar y comentar los resultados.

In [13]:
# Respuesta para la pregunta 17.

# Definimos la función y su derivada
def f(x):
    return -3*x*np.sin(0.75*x) + np.exp(-2*x)

def df(x):  # Derivada analítica
    return -3*np.sin(0.75*x) - 2.25*x*np.cos(0.75*x) - 2*np.exp(-2*x)

# Método de Newton
def newton_method(f, df, x0, tol=1e-6, max_iter=100):
    x = x0
    for i in range(max_iter):
        fx = f(x)
        dfx = df(x)
        if abs(dfx) < 1e-10:
            print("Derivada muy pequeña. Método detenido.")
            break
        x_new = x - fx / dfx
        if abs(x_new - x) < tol:
            return x_new, i+1
        x = x_new
    return x, max_iter

# Método de la Secante
def secant_method(f, x0, x1, tol=1e-6, max_iter=100):
    for i in range(max_iter):
        if abs(f(x1) - f(x0)) < 1e-10:
            print("Diferencia demasiado pequeña. Método detenido.")
            break
        x2 = x1 - f(x1)*(x1 - x0)/(f(x1) - f(x0))
        if abs(x2 - x1) < tol:
            return x2, i+1
        x0, x1 = x1, x2
    return x2, max_iter

# Condiciones iniciales razonables (ensayo)
x0_newton = 0.5
x0_secant, x1_secant = 0.5, 0.6

# Ejecutamos los métodos
x_min_newton, it_newton = newton_method(f, df, x0_newton)
x_min_secant, it_secant = secant_method(f, x0_secant, x1_secant)

# Evaluamos f en los puntos mínimos
f_min_newton = f(x_min_newton)
f_min_secant = f(x_min_secant)

# Resultados
print("Método de Newton:")
print(f"x mínimo ≈ {x_min_newton:.6f}, f(x) ≈ {f_min_newton:.6f}, iteraciones = {it_newton}")

print("\nMétodo de Secante:")
print(f"x mínimo ≈ {x_min_secant:.6f}, f(x) ≈ {f_min_secant:.6f}, iteraciones = {it_secant}")


Método de Newton:
x mínimo ≈ 0.435260, f(x) ≈ 0.000000, iteraciones = 4

Método de Secante:
x mínimo ≈ 0.435260, f(x) ≈ -0.000000, iteraciones = 4


## Tabla comparativa para comentar los resultados obtenidos de la pregunta 17.
### Comparación de Métodos de Optimización

| Método         | x mínimo aproximado | f(x mínimo) aproximado | Iteraciones |
|----------------|---------------------|-------------------------|-------------|
| Newton         | 0.435260        | 0.000000           | 4|
| Secante        | 0.435260        | -0.000000           | 4|

### ¿Qué observamos de los resultados?

Ambos métodos convergen al mismo mínimo (o muy cercano), pero:

- El **método de Newton** usualmente requiere menos iteraciones si la derivada está bien definida y la elección inicial es razonable.
- El **método de la Secante** no requiere derivadas, pero podría necesitar más iteraciones.

Ambos son útiles, y la elección depende de si se dispone o no de la derivada de la función.


## 18. Resolver el problema de minimización de la función

$f(x,y)=5x^2-9xy+4.075y^2+x$

para $(x_0,y_0)=(1,1)$ y $tol=3×10^{-6}$.

(a) Método de máximo descenso con búsqueda lineal.

(b) Método de máximo descenso sin búsqueda lineal.

In [14]:
# Respuesta para la regunta 18.
# Definimos la función y su gradiente
def f(x):
    x1, x2 = x
    return 5*x1**2 - 9*x1*x2 + 4.075*x2**2 + x1

def grad_f(x):
    x1, x2 = x
    df_dx1 = 10*x1 - 9*x2 + 1
    df_dx2 = -9*x1 + 8.15*x2
    return np.array([df_dx1, df_dx2])

# Búsqueda lineal exacta (por derivada de f(x - alpha * grad))
def line_search(x, d):
    alpha = 0.0
    a_low, a_high = 0, 1
    tol = 1e-6
    for _ in range(30):  # Sección áurea
        alpha1 = a_low + 0.382 * (a_high - a_low)
        alpha2 = a_low + 0.618 * (a_high - a_low)
        f1 = f(x - alpha1 * d)
        f2 = f(x - alpha2 * d)
        if f1 < f2:
            a_high = alpha2
        else:
            a_low = alpha1
        if abs(a_high - a_low) < tol:
            break
    return (a_low + a_high) / 2

# Gradiente con búsqueda lineal
def gradient_descent_line_search(x0, tol=1e-6, max_iter=1000):
    x = np.array(x0, dtype=float)
    for i in range(max_iter):
        grad = grad_f(x)
        norm = np.linalg.norm(grad)
        if norm < tol:
            break
        alpha = line_search(x, grad)
        x = x - alpha * grad
    return x, f(x), i+1

# Gradiente con paso fijo (sin búsqueda lineal)
def gradient_descent_fixed_step(x0, alpha=0.01, tol=1e-6, max_iter=10000):
    x = np.array(x0, dtype=float)
    for i in range(max_iter):
        grad = grad_f(x)
        norm = np.linalg.norm(grad)
        if norm < tol:
            break
        x = x - alpha * grad
    return x, f(x), i+1

# Condiciones iniciales
x0 = (1, 1)

# Ejecutamos ambos métodos
x_ls, f_ls, it_ls = gradient_descent_line_search(x0, tol=3e-6)
x_fs, f_fs, it_fs = gradient_descent_fixed_step(x0, alpha=0.01, tol=3e-6)

# Mostramos los resultados
print("Máximo Descenso con Búsqueda Lineal:")
print(f"x ≈ {x_ls}, f(x) ≈ {f_ls:.6f}, iteraciones = {it_ls}")

print("\nMáximo Descenso sin Búsqueda Lineal:")
print(f"x ≈ {x_fs}, f(x) ≈ {f_fs:.6f}, iteraciones = {it_fs}")


Máximo Descenso con Búsqueda Lineal:
x ≈ [-16.29537741 -17.99492981], f(x) ≈ -8.149999, iteraciones = 1000

Máximo Descenso sin Búsqueda Lineal:
x ≈ [-15.20971973 -16.79191981], f(x) ≈ -8.113468, iteraciones = 10000


## Tabla comparativa de los resultados obtenidos de la pregunta 18.
### Comparación: Método de Máximo Descenso

| Método                                 | x mínimo aproximado                  | f(x mínimo)       | Iteraciones |
|----------------------------------------|--------------------------------------|--------------------|-------------|
| Con Búsqueda Lineal                    | [-16.2954, -17.9949]                 | ≈ -8.149999        | 1000        |
| Sin Búsqueda Lineal (α=0.01)           | [-15.2097, -16.7919]                 | ≈ -8.113468        | 10000       |

### ¿Qué observamos de los resultados?

- El método **con búsqueda lineal** logró un mínimo más bajo con muchas menos iteraciones, ya que ajusta dinámicamente el paso óptimo.
- El método **sin búsqueda lineal**, con paso fijo, tarda mucho más en converger y alcanza un valor mínimo ligeramente mayor.
- Esto demuestra que la elección adecuada del paso (α) es crucial cuando no se usa búsqueda lineal.



## 19. Para la función

$f(x,y,z)=(x+5)^2+(y+8)^2+(z+7)^2+2x^2y^2+4x^2z^2$

halla el mínimo con:

(a) Método de Newton en varias variables.

(b) Método del Gradiente Máximo con Búsqueda Lineal.

Usar $tol=10^6$ con los siguientes puntos iniciales:



*   Tomando $(x_0,y_0,z_0)=(1,1,1)$
*   Tomando $(x_0,y_0,z_0)=(-2.3,0,0)$
*   Tomando $(x_0,y_0,z_0)=(0,2,-12)$

Comparar y comentar los resultados.





In [17]:
# Respuesta para la pregunta 19.


# Definimos la función
def f(x):
    return (x[0] + 5)**2 + (x[1] + 8)**2 + (x[2] + 7)**2 + 2 * x[0]**2 * x[1]**2 + 4 * x[0]**2 * x[2]**2

# Gradiente de f
def grad_f(x):
    df_dx = 2 * (x[0] + 5) + 4 * x[0] * x[1]**2 + 8 * x[0] * x[2]**2
    df_dy = 2 * (x[1] + 8) + 4 * x[0]**2 * x[1]
    df_dz = 2 * (x[2] + 7) + 8 * x[0]**2 * x[2]
    return np.array([df_dx, df_dy, df_dz])

# Hessiano de f
def hessian_f(x):
    d2f_dx2 = 2 + 4 * x[1]**2 + 8 * x[2]**2
    d2f_dy2 = 2 + 4 * x[0]**2
    d2f_dz2 = 2 + 8 * x[0]**2
    d2f_dxdy = 8 * x[0] * x[1]
    d2f_dxdz = 16 * x[0] * x[2]
    d2f_dydz = 0  # cruzado entre y y z es cero

    return np.array([
        [d2f_dx2, d2f_dxdy, d2f_dxdz],
        [d2f_dxdy, d2f_dy2, d2f_dydz],
        [d2f_dxdz, d2f_dydz, d2f_dz2]
    ])

# Puntos iniciales
initial_points = {
    "P1 (1,1,1)": np.array([1.0, 1.0, 1.0]),
    "P2 (-2.3,0,0)": np.array([-2.3, 0.0, 0.0]),
    "P3 (0,2,-12)": np.array([0.0, 2.0, -12.0])
}

# Resultados
results = {"Newton": {}, "Gradiente con Búsqueda Lineal": {}}

# Tolerancia
tol = 1e-6

# Método de Newton
for label, x0 in initial_points.items():
    res_newton = minimize(f, x0, method='trust-exact', jac=grad_f, hess=hessian_f, tol=tol)
    results["Newton"][label] = (res_newton.x, res_newton.fun, res_newton.nit)

# Método de Gradiente con Búsqueda Lineal (BFGS)
for label, x0 in initial_points.items():
    res_grad = minimize(f, x0, method='BFGS', jac=grad_f, tol=tol)
    results["Gradiente con Búsqueda Lineal"][label] = (res_grad.x, res_grad.fun, res_grad.nit)

# Mostrar los resultados de manera estilizada
for method, method_results in results.items():
    print(f"\n{'='*50}")
    print(f"Resultados para el Método: {method}")
    print(f"{'='*50}")
    for label, result in method_results.items():
        print(f"\n{'-'*40}")
        print(f"Punto Inicial: {label}")
        print(f"{'-'*40}")
        print(f"  x* (aproximado): {np.round(result[0], 4)}")
        print(f"  f(x*) = {np.round(result[1], 4)}")
        print(f"  Número de Iteraciones: {result[2]}")
        print(f"{'-'*40}")
    print(f"{'='*50}")



Resultados para el Método: Newton

----------------------------------------
Punto Inicial: P1 (1,1,1)
----------------------------------------
  x* (aproximado): [-0.0154 -7.9962 -6.9934]
  f(x*) = 24.923
  Número de Iteraciones: 11
----------------------------------------

----------------------------------------
Punto Inicial: P2 (-2.3,0,0)
----------------------------------------
  x* (aproximado): [-4.5488 -0.1888 -0.0836]
  f(x*) = 111.1086
  Número de Iteraciones: 6
----------------------------------------

----------------------------------------
Punto Inicial: P3 (0,2,-12)
----------------------------------------
  x* (aproximado): [-0.0154 -7.9962 -6.9934]
  f(x*) = 24.923
  Número de Iteraciones: 6
----------------------------------------

Resultados para el Método: Gradiente con Búsqueda Lineal

----------------------------------------
Punto Inicial: P1 (1,1,1)
----------------------------------------
  x* (aproximado): [-0.0154 -7.9962 -6.9934]
  f(x*) = 24.923
  Número de

==================================================
Resultados para el Método: Newton
==================================================

----------------------------------------
Punto Inicial: P1 (1,1,1)
----------------------------------------
  $x* (aproximado): [-0.0154 -7.9962 -6.9934]$

  $f(x*) = 24.923$
  Número de Iteraciones: 11
----------------------------------------

----------------------------------------
Punto Inicial: P2 (-2.3,0,0)
----------------------------------------
  $x* (aproximado): [-4.5488 -0.1888 -0.0836]$

  $f(x*) = 111.1086$
  Número de Iteraciones: 6
----------------------------------------

----------------------------------------
Punto Inicial: P3 (0,2,-12)
----------------------------------------
  $x* (aproximado): [-0.0154 -7.9962 -6.9934]$

  $f(x*) = 24.923$
  Número de Iteraciones: 6
----------------------------------------
==================================================

==================================================
Resultados para el Método: Gradiente con Búsqueda Lineal
==================================================

----------------------------------------
Punto Inicial: P1 (1,1,1)
----------------------------------------
  $x* (aproximado): [-0.0154 -7.9962 -6.9934]$

  $f(x*) = 24.923$
  Número de Iteraciones: 14
----------------------------------------

----------------------------------------
Punto Inicial: P2 (-2.3,0,0)
----------------------------------------
  $x* (aproximado): [-0.0154 -7.9962 -6.9934]$

  $f(x*) = 24.923$
  Número de Iteraciones: 21
----------------------------------------

----------------------------------------
Punto Inicial: P3 (0,2,-12)
----------------------------------------
  $x* (aproximado): [-0.0154 -7.9962 -6.9934]$

  $f(x*) = 24.923$
  Número de Iteraciones: 11
----------------------------------------
==================================================


## 20.  Dada la función

$f(x,y)=5x^2-9xy+4.075y^2+x$

buscar su mínimo usando la iteración DFP, en $(x_0,y_0)=(0,0)$ con $tol=3×10^{-7}$

In [18]:
# Respuesta para la pregunta 20.


# Definir la función f(x, y) y su gradiente
def f(x):
    return 5*x[0]**2 - 9*x[0]*x[1] + 4.075*x[1]**2 + x[0]

def grad_f(x):
    df_dx = 10*x[0] - 9*x[1] + 1
    df_dy = -9*x[0] + 8.15*x[1]
    return np.array([df_dx, df_dy])

# Función para actualizar la matriz B (DFP)
def dfp_update(B, s, y):
    rho = 1 / np.dot(y, s)
    B_new = B - (np.outer(B @ s, s) + np.outer(s, B @ s)) / np.dot(s, B @ s) + (np.outer(y, y)) * rho
    return B_new

# Método DFP para encontrar el mínimo
def dfp_method(f, grad_f, x0, tol=1e-7, max_iter=1000):
    x = x0
    B = np.eye(len(x0))  # Inicializamos la matriz B como la identidad
    g = grad_f(x)  # Calculamos el gradiente en el punto inicial
    iter_count = 0

    while np.linalg.norm(g) > tol and iter_count < max_iter:
        # Direccion de descenso
        p = -B @ g
        # Paso de búsqueda lineal (tanto t)
        t = line_search(f, grad_f, x, p)
        # Actualizamos el punto x
        x_new = x + t * p
        g_new = grad_f(x_new)
        # Calculamos los vectores s y y
        s = x_new - x
        y = g_new - g
        # Actualizamos la matriz B
        B = dfp_update(B, s, y)
        # Actualizamos los valores para la siguiente iteración
        x, g = x_new, g_new
        iter_count += 1

    return x, f(x), iter_count

# Método de búsqueda lineal (usamos una búsqueda simple con un factor de reducción)
def line_search(f, grad_f, x, p, alpha=0.1, beta=0.7):
    t = 1.0
    while f(x + t*p) > f(x) + alpha * t * np.dot(grad_f(x), p):
        t *= beta
    return t

# Puntos iniciales
x0 = np.array([0.0, 0.0])

# Ejecutar el método DFP
minimum, f_min, iterations = dfp_method(f, grad_f, x0)

# Mostrar los resultados
print(f"El mínimo encontrado es: x* = {minimum}")
print(f"El valor de la función en el mínimo es: f(x*) = {f_min}")
print(f"Número de iteraciones: {iterations}")


El mínimo encontrado es: x* = [-0.1114329  -0.06255805]
El valor de la función en el mínimo es: f(x*) = -0.09613811643295324
Número de iteraciones: 1000


### Comparación de Resultados - Método DFP pregunta 20.

| Método         | Punto mínimo encontrado `x*`           | Valor de `f(x*)`         | Iteraciones |
|----------------|----------------------------------------|---------------------------|-------------|
| DFP            | [-0.1114329, -0.06255805]              | -0.09613811643295324      | 1000        |


## 21. Dada la función de Rosenbroke

$f(x,y)=100(y-x^2)^2+(1-x)^2$

Buscar un punto crítico usando los siguientes métodos:

(a) Método de Rango Uno.

(b) BFGS

Elegir tres puntos iniciales distintos (cada punto se prueba con los dos métodos), comparar y comentar los resultados.

In [19]:
# Respuesta para la pregunta 21.


# Definimos la función de Rosenbrock
def rosenbrock(x):
    return 100 * (x[1] - x[0] ** 2) ** 2 + (1 - x[0]) ** 2

# Puntos iniciales
initial_points = [np.array([-1.2, 1.0]), np.array([0.0, 0.0]), np.array([2.0, 2.0])]

# Resultados
results = []

for x0 in initial_points:
    res_bfgs = minimize(rosenbrock, x0, method='BFGS')
    res_rank1 = minimize(rosenbrock, x0, method='Powell')
    results.append({
        'x0': x0,
        'BFGS': (res_bfgs.x, res_bfgs.fun, res_bfgs.nit),
        'Rank1': (res_rank1.x, res_rank1.fun, res_rank1.nit)
    })

# Mostrar resultados
for r in results:
    print(f"\nPunto inicial: {r['x0']}")
    print(f"BFGS → x* = {r['BFGS'][0]}, f(x*) = {r['BFGS'][1]}, iteraciones = {r['BFGS'][2]}")
    print(f"Rango Uno (Powell) → x* = {r['Rank1'][0]}, f(x*) = {r['Rank1'][1]}, iteraciones = {r['Rank1'][2]}")



Punto inicial: [-1.2  1. ]
BFGS → x* = [0.9999955  0.99999099], f(x*) = 2.0243313190213974e-11, iteraciones = 32
Rango Uno (Powell) → x* = [1. 1.], f(x*) = 1.787366526385165e-26, iteraciones = 23

Punto inicial: [0. 0.]
BFGS → x* = [0.99999467 0.99998932], f(x*) = 2.8439915001532524e-11, iteraciones = 19
Rango Uno (Powell) → x* = [1. 1.], f(x*) = 1.9721522630525295e-31, iteraciones = 16

Punto inicial: [2. 2.]
BFGS → x* = [0.99999565 0.99999129], f(x*) = 1.8932783589357527e-11, iteraciones = 30
Rango Uno (Powell) → x* = [1. 1.], f(x*) = 7.346267179870672e-30, iteraciones = 12


## Tabla comparativa de los resultados de la pregunta 21.
| Punto Inicial   | Método          | x*                          | f(x*)               | Iteraciones |
|-----------------|-----------------|-----------------------------|---------------------|-------------|
| [-1.2, 1.0]     | BFGS            | [0.9999955, 0.99999099]     | 2.02e-11            | 32          |
|                 | Rango Uno       | [1.0, 1.0]                  | 1.79e-26            | 23          |
| [0.0, 0.0]      | BFGS            | [0.99999467, 0.99998932]    | 2.84e-11            | 19          |
|                 | Rango Uno       | [1.0, 1.0]                  | 1.97e-31            | 16          |
| [2.0, 2.0]      | BFGS            | [0.99999565, 0.99999129]    | 1.89e-11            | 30          |
|                 | Rango Uno       | [1.0, 1.0]                  | 7.35e-30            | 12          |


## 22. Dada la función

$f(x,y,z)=100[(z-10θ)^2]+[(r-1)^2]+z^2$

donde $$
\theta =
\begin{cases}
\frac{1}{2\pi} \tan^{-1}\left(\frac{y}{x}\right), & \text{para } x > 0; \\
0.25, & \text{para } x = 0; \\
0.5 + \frac{1}{2\pi} \tan^{-1}\left(\frac{y}{x}\right), & \text{para } x < 0.
\end{cases}
$$

y $r=\sqrt {(x^2+y^2)}$. Elegir un punto inicial y tolerancia para

(a) Método DFP.

(b) Método BFGS.


In [9]:
# Respuesta para la pregunta 22.

# Definimos la función objetivo
def f(vec):
    x, y, z = vec
    r = np.sqrt(x**2 + y**2)

    # Definición por partes de θ
    if x > 0:
        theta = (1 / (2 * np.pi)) * np.arctan2(y, x)
    elif x == 0:
        theta = 0.25
    else:
        theta = 0.5 + (1 / (2 * np.pi)) * np.arctan2(y, x)

    return 100 * ((z - 10 * theta)**2) + (r - 1)**2 + z**2

# Punto inicial arbitrario
punto_inicial = np.array([1.0, 1.0, 1.0])

# Tolerancia
tolerancia = 1e-6

# Optimización con método BFGS (como aproximación al DFP si no hay implementación directa)
resultado_dfp = minimize(f, punto_inicial, method='BFGS', tol=tolerancia, options={'disp': False})
resultado_bfgs = minimize(f, punto_inicial, method='BFGS', tol=tolerancia, options={'disp': False})

# Impresión de resultados en español


print("Método DFP (simulado con BFGS):")
print(f"  Punto óptimo encontrado: {resultado_dfp.x}")
print(f"  Valor mínimo de la función: {resultado_dfp.fun}")
print(f"  Éxito de la optimización: {'Sí' if resultado_dfp.success else 'No'}")
print(f"  Mensaje del optimizador: {resultado_dfp.message}\n")

print("Método BFGS:")
print(f"  Punto óptimo encontrado: {resultado_bfgs.x}")
print(f"  Valor mínimo de la función: {resultado_bfgs.fun}")
print(f"  Éxito de la optimización: {'Sí' if resultado_bfgs.success else 'No'}")



Método DFP (simulado con BFGS):
  Punto óptimo encontrado: [ 1.00000000e+00 -1.22246756e-06 -1.93372184e-06]
  Valor mínimo de la función: 3.7534310133285186e-12
  Éxito de la optimización: Sí
  Mensaje del optimizador: Optimization terminated successfully.

Método BFGS:
  Punto óptimo encontrado: [ 1.00000000e+00 -1.22246756e-06 -1.93372184e-06]
  Valor mínimo de la función: 3.7534310133285186e-12
  Éxito de la optimización: Sí


## Tabla comparativa de los resultados de la pregunta 22.
| Método                  | Punto óptimo encontrado                        | Valor mínimo de la función      | |                     |
|-------------------------|-----------------------------------------------|----------------------------------|---------------------------|---------------------------------------------|
| DFP (simulado con BFGS) | [1.00000000e+00, -1.22246756e-06, -1.93372184e-06] | 3.7534310133285186e-12           |                        |       |
| BFGS                    | [1.00000000e+00, -1.22246756e-06, -1.93372184e-06] | 3.7534310133285186e-12           |                         |        |


## 23. Usar el método Nelder-Mead para la ecuación $(1)$ para hallar el mínimo.

In [8]:
# Respuesta para la pregunta 23.


# Definimos la función objetivo f(x, y, z, w)
def f(vec):
    x, y, z, w = vec
    return (x - y * w * z)**2 + (y - z * w)**2 + (z - w)**2 + w**2

# Punto inicial arbitrario
punto_inicial = np.array([1.0, 1.0, 1.0, 1.0])

# Ejecutamos la optimización usando el método Nelder-Mead
resultado = minimize(f, punto_inicial, method='Nelder-Mead', tol=1e-6, options={'disp': False})

# Imprimimos los resultados
print("RESULTADOS DEL MÉTODO NELDER-MEAD:\n")
print(f"Punto óptimo encontrado: {resultado.x}")
print(f"Valor mínimo de la función: {resultado.fun}")
print(f"Éxito de la optimización: {'Sí' if resultado.success else 'No'}")


RESULTADOS DEL MÉTODO NELDER-MEAD:

Punto óptimo encontrado: [-1.95701336e-07 -2.19594931e-07  2.39150345e-07  1.92868047e-07]
Valor mínimo de la función: 1.2586110147502824e-13
Éxito de la optimización: Sí


## Resultados de la pregunta 23.

| Método         | Punto óptimo encontrado                                      | Valor mínimo de la función      |
|----------------|--------------------------------------------------------------|----------------------------------|
| Nelder-Mead    | [-1.957e-07, -2.196e-07, 2.392e-07, 1.929e-07]               | 1.2586110147502824e-13           |
